In [ ]:
#Train: IG, Test: NIOS

In [1]:
import pandas as pd

# Load each set separately
train_df = pd.read_excel("train_ig.xlsx")
val_df = pd.read_excel("validset.xlsx")
test_df = pd.read_excel("testset.xlsx")

# Check sizes
print("Train size:", len(train_df))
print("Validation size:", len(val_df))
print("Test size:", len(test_df))


Train size: 10101
Validation size: 2851
Test size: 2851


In [2]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
import sacrebleu
from tqdm import tqdm
import pandas as pd

In [3]:
# Load NLLB model
model_name = "facebook/nllb-200-distilled-1.3B"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

In [4]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)
test_dataset = Dataset.from_pandas(test_df)

src_lang = "san_Deva"   # Sanskrit
tgt_lang = "hin_Deva"   # Hindi
tokenizer.src_lang = src_lang
tokenizer.tgt_lang = tgt_lang

def preprocess_function(examples):
    inputs = [str(x) if x is not None else "" for x in examples["san"]]
    targets = [str(x) if x is not None else "" for x in examples["hin"]]
    #inputs = examples["san"]
    #targets = examples["hin"]
    model_inputs = tokenizer(inputs, max_length=128, truncation=True)
    labels = tokenizer(text_target=targets, max_length=128, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_tokenized = train_dataset.map(preprocess_function, batched=True)
val_tokenized = val_dataset.map(preprocess_function, batched=True)

Map:   0%|          | 0/10101 [00:00<?, ? examples/s]

Map:   0%|          | 0/2851 [00:00<?, ? examples/s]

In [12]:
test_tokenized = test_dataset.map(preprocess_function, batched=True)

Map:   0%|          | 0/2851 [00:00<?, ? examples/s]

In [5]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

In [13]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./results",
    do_eval=True,           # instead of evaluation_strategy
    save_total_limit=2,
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=10,
    weight_decay=0.01,
    predict_with_generate=True,
    logging_dir='./logs',
)


In [7]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

In [14]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=test_tokenized,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

/tmp/ipykernel_1268526/3896070871.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


In [15]:
trainer.train()

Step,Training Loss
500,1.424900
1000,1.605700
1500,1.503800
2000,1.474500
2500,1.455100
3000,1.186400
3500,1.114400
4000,1.140300
4500,1.115500
5000,1.141300


/nlsasfs/home/dibd/dibd-coild/igtduw/kausha/Conda/lib/python3.13/site-packages/transformers/modeling_utils.py:3918: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 200}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


TrainOutput(global_step=25260, training_loss=0.6550566494323296, metrics={'train_runtime': 6028.1321, 'train_samples_per_second': 16.756, 'train_steps_per_second': 4.19, 'total_flos': 1.4634201705357312e+16, 'train_loss': 0.6550566494323296, 'epoch': 10.0})

In [16]:
import numpy as np
from evaluate import load

# Load metrics
bleu = load("sacrebleu")
chrf = load("chrf")

# Generate predictions
preds = trainer.predict(test_tokenized)

# Step 1: Ensure proper decoding
if preds.predictions.ndim == 3:  # sometimes logits instead of token IDs
    preds_ids = np.argmax(preds.predictions, axis=-1)
else:
    preds_ids = preds.predictions

# Step 2: Clip invalid values to tokenizer vocab size
vocab_size = len(tokenizer)
preds_ids = np.clip(preds_ids, 0, vocab_size - 1)

# Step 3: Decode safely
decoded_preds = tokenizer.batch_decode(preds_ids, skip_special_tokens=True)
decoded_labels = tokenizer.batch_decode(np.where(preds.label_ids != -100, preds.label_ids, tokenizer.pad_token_id),
                                        skip_special_tokens=True)

# Step 4: Compute metrics
bleu_score = bleu.compute(predictions=decoded_preds, references=[[l] for l in decoded_labels])
chrf_score = chrf.compute(predictions=decoded_preds, references=[[l] for l in decoded_labels])

print(f"BLEU score: {bleu_score['score']:.2f}")
print(f"chrF score: {chrf_score['score']:.2f}")


BLEU score: 18.34
chrF score: 45.45


In [22]:
# COMPLETE EVALUATION: METEOR + BERTScore + COMET

import os
import nltk
from nltk.translate.meteor_score import meteor_score
from bert_score import score as bert_score
from comet import download_model, load_from_checkpoint


# Force COMET to run on CPU (old API compatible)
#os.environ["CUDA_VISIBLE_DEVICES"] = ""

# Download METEOR resources (run once)
nltk.download("wordnet")
nltk.download("omw-1.4")

refs = decoded_labels   # reference Hindi sentences
hyps = decoded_preds    # model predictions

assert isinstance(refs, list) and isinstance(hyps, list)
assert len(refs) == len(hyps)

def tokenize(sent):
    return sent.strip().split()

# METEOR
meteor_scores = [
    meteor_score(
        [tokenize(ref)],
        tokenize(hyp)
    )
    for ref, hyp in zip(refs, hyps)
]

meteor = sum(meteor_scores) / len(meteor_scores)
print(f"METEOR: {meteor:.4f}")

# BERTScore (Hindi)
P, R, F1 = bert_score(
    hyps,
    refs,
    lang="hi",
    rescale_with_baseline=True,
    verbose=False
)

print(f"BERT Precision: {P.mean().item():.4f}")
print(f"BERT Recall:    {R.mean().item():.4f}")
print(f"BERT F1:        {F1.mean().item():.4f}")

# COMET (CPU | old API safe)
model_path = download_model("Unbabel/wmt22-comet-da")
comet_model = load_from_checkpoint(model_path)

# COMET input format
data = [
    {
        "src": "",
        "mt": hyp,
        "ref": ref
    }
    for hyp, ref in zip(hyps, refs)
]

comet_output = comet_model.predict(
    data,
    batch_size=4
)

comet_final = sum(comet_output["scores"]) / len(comet_output["scores"])
print(f"COMET: {comet_final:.4f}")


[nltk_data] Downloading package wordnet to /nlsasfs/home/dibd/dibd-
[nltk_data]     coild/igtduw/kausha/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /nlsasfs/home/dibd/dibd-
[nltk_data]     coild/igtduw/kausha/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


METEOR: 0.3781


BERT Precision: 0.8337
BERT Recall:    0.8293
BERT F1:        0.8313


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.6.0. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../.cache/huggingface/hub/models--Unbabel--wmt22-comet-da/snapshots/2760a223ac957f30acfb18c8aa649b01cf1d75f2/checkpoints/model.ckpt`
Encoder model frozen.
/nlsasfs/home/dibd/dibd-coild/igtduw/kausha/Conda/lib/python3.13/site-packages/pytorch_lightning/core/saving.py:197: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
/nlsasfs/home/dibd/dibd-coild/igtduw/kausha/Conda/lib/python3.13/site-packages/torch/__init__.py:1551: UserWarning: Please use the new 

COMET: 0.6685
